# 07 — Faster R-CNN Training v2 (Resolution + RFS + Augmentation)


In [1]:
from pathlib import Path

_here = Path.cwd().resolve()
for _root in (_here, *_here.parents):
    if (_root / "src").is_dir() and (_root / "outputs").is_dir():
        break
else:
    raise FileNotFoundError("Repo root not found (need src/ and outputs/).")

print(f"Repo root: {_root}")

Repo root: C:\Users\micha\Downloads\Object-Detection-main


In [2]:
import sys
import os
import numpy as np
import torch
from torch.utils.data import DataLoader

project_path = str(_root)
if project_path not in sys.path:
    sys.path.append(project_path)
    sys.path.append(os.path.join(project_path, 'src'))

from src.utils import SEED, CLASS_MAP, FRCNN_CLASS_MAP, NUM_CLASSES, seed_everything, log_environment
from src.fasterrcnn_dataset import BDD100KDataset
from src.fasterrcnn_utils import collate_fn, build_fasterrcnn, train_one_epoch, val_one_epoch
from src.repeat_factor_sampler import build_repeat_factor_sampler

## 1. Environment & Seed

In [3]:
seed_everything(SEED)
log_environment()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"FRCNN_CLASS_MAP: {FRCNN_CLASS_MAP}")
print(f"NUM_CLASSES: {NUM_CLASSES}")

PyTorch: 2.11.0+cu128
Ultralytics: 8.4.42
GPU: NVIDIA GeForce RTX 5090
CUDA: 12.8
Device: cuda
FRCNN_CLASS_MAP: {'car': 1, 'person': 2, 'truck': 3, 'bus': 4, 'motor': 5, 'bike': 6, 'traffic light': 7, 'traffic sign': 8}
NUM_CLASSES: 9


## 2. Dataset & DataLoaders

In [4]:
DATASET_ROOT = _root / "outputs" / "bdd100k_preprocessing"

TRAIN_IMG_DIR = f"{DATASET_ROOT}/bdd100k-yolo-subset-v1/images/train"
VAL_IMG_DIR = f"{DATASET_ROOT}/bdd100k-yolo-subset-v1/images/val"

train_dataset = BDD100KDataset(
    image_dir=TRAIN_IMG_DIR,
    annotation_file=f"{DATASET_ROOT}/train_annotations.json",
    class_map=FRCNN_CLASS_MAP,
)

val_dataset = BDD100KDataset(
    image_dir=VAL_IMG_DIR,
    annotation_file=f"{DATASET_ROOT}/val_annotations.json",
    class_map=FRCNN_CLASS_MAP,
)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

Train samples: 7000
Val samples: 1500


## 3. Repeat Factor Sampling

To address minority class imbalance by upsampling images containing rare classes (motor, bike).

Using `repeat_thresh=0.05` - classes appearing in <5% of images get upsampled.

In [5]:
sampler = build_repeat_factor_sampler(
    dataset=train_dataset,
    num_classes=NUM_CLASSES,
    repeat_thresh=0.05,
    verbose=True,
)

Computing class frequencies for repeat factor sampler...

Class sampling multipliers:
  car: freq=0.9911 -> multiplier=1.00x
  person: freq=0.3144 -> multiplier=1.00x
  truck: freq=0.2743 -> multiplier=1.00x
  bus: freq=0.1200 -> multiplier=1.00x
  motor: freq=0.0324 -> multiplier=1.24x
  bike: freq=0.0567 -> multiplier=1.00x
  traffic light: freq=0.5651 -> multiplier=1.00x
  traffic sign: freq=0.8154 -> multiplier=1.00x
  class_9: freq=0.0000 -> multiplier=1.00x


## 4. Model Setup with Resolution Scaling

min_size=1024, max_size=1600 (up from 800/1333)
Gives FPN more pixels for small object detection.

In [6]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.transform import GeneralizedRCNNTransform

def build_fasterrcnn_v2(num_classes):
    model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
    
    model.transform = GeneralizedRCNNTransform(
        min_size=1024,   # up from 800
        max_size=1600,   # up from 1333
        image_mean=[0.485, 0.456, 0.406],
        image_std=[0.229, 0.224, 0.225],
    )
    
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

In [7]:
model = build_fasterrcnn_v2(num_classes=NUM_CLASSES)
model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
print(f"Trainable parameters: {sum(p.numel() for p in params):,}")

Trainable parameters: 41,112,636


## 5. Optimizer & Scheduler

In [8]:
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

print("Optimizer: SGD (lr=0.005, momentum=0.9, weight_decay=0.0005)")
print("Scheduler: StepLR (step_size=5, gamma=0.1)")

Optimizer: SGD (lr=0.005, momentum=0.9, weight_decay=0.0005)
Scheduler: StepLR (step_size=5, gamma=0.1)


## 6. DataLoaders with RFS

In [9]:
BATCH_SIZE = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=2,
    collate_fn=collate_fn
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

Train batches: 1750
Val batches: 375


## 7. Training Loop

In [10]:
NUM_EPOCHS = 20
PATIENCE = 5

best_val_loss = float("inf")
epochs_no_improve = 0
history = {
    "train_loss": [],
    "val_loss": [],
}

In [11]:
import json
import shutil

OUTPUT_DIR = _root / "outputs" / "bdd100k_project"
BEST_MODEL_PATH = OUTPUT_DIR / "fasterrcnn_v2_best.pth"
HISTORY_PATH = OUTPUT_DIR / "fasterrcnn_v2_loss_history.json"

shutil.copyobj = lambda src, dst: shutil.copy2(src, dst)
torch.save(model.state_dict(), BEST_MODEL_PATH)
print(f"Initial model saved to {BEST_MODEL_PATH}")

Initial model saved to C:\Users\micha\Downloads\Object-Detection-main\outputs\bdd100k_project\fasterrcnn_v2_best.pth


In [12]:
for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    
    train_loss, _ = train_one_epoch(model, optimizer, train_loader, device, epoch)
    val_loss = val_one_epoch(model, val_loader, device, epoch)
    
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    
    print(f"Epoch {epoch + 1} | LR={current_lr:.6f} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"  -> Best model saved (val_loss={val_loss:.4f})")
    else:
        epochs_no_improve += 1
        print(f"  -> No improvement ({epochs_no_improve}/{PATIENCE})")
    
    if epochs_no_improve >= PATIENCE:
        print(f"\nEarly stopping triggered after {epoch + 1} epochs")
        break


Epoch 1/20


Val Epoch 0: 100%|██████████| 375/375 [00:42<00:00,  8.89it/s]


Epoch 1 | LR=0.005000 | Train Loss: 0.9382 | Val Loss: 0.8826
  -> Best model saved (val_loss=0.8826)

Epoch 2/20


Val Epoch 1: 100%|██████████| 375/375 [00:41<00:00,  9.08it/s]


Epoch 2 | LR=0.005000 | Train Loss: 0.8348 | Val Loss: 0.8950
  -> No improvement (1/5)

Epoch 3/20


...

Epoch 9 | LR=0.000500 | Train Loss: 0.6177 | Val Loss: 0.9104
  -> No improvement (3/5)

Epoch 10/20


Val Epoch 9: 100%|██████████| 375/375 [00:43<00:00,  8.64it/s]


Epoch 10 | LR=0.000050 | Train Loss: 0.6083 | Val Loss: 0.9043
  -> No improvement (4/5)

Epoch 11/20


Val Epoch 10: 100%|██████████| 375/375 [00:41<00:00,  8.99it/s]

Epoch 11 | LR=0.000050 | Train Loss: 0.5953 | Val Loss: 0.9049
  -> No improvement (5/5)

Early stopping triggered after 11 epochs


In [13]:
with open(HISTORY_PATH, 'w') as f:
    json.dump(history, f, indent=2)

print(f"\nTraining complete!")
print(f"Best model: {BEST_MODEL_PATH}")
print(f"History: {HISTORY_PATH}")
print(f"Best val_loss: {best_val_loss:.4f}")


Training complete!
Best model: C:\Users\micha\Downloads\Object-Detection-main\outputs\bdd100k_project\fasterrcnn_v2_best.pth
History: C:\Users\micha\Downloads\Object-Detection-main\outputs\bdd100k_project\fasterrcnn_v2_loss_history.json
Best val_loss: 0.8688
